# STAR — Kaggle: train v3c trên 50K (smart-sample + ViTPose, text FROZEN) → best.pth

Y HỆT công thức **v3c** (đã ra 0.7485 ở 10K), chỉ đổi data **10K → 50K**:
- **smart-sample** `group_by=pair` · **text BERT 0-6 ĐÓNG BĂNG** (`lora_freeze_text=true`, KHÔNG LoRA text) ·
  **ViTPose** (`pose_enabled=true`) · LoRA r=16 · λ_smoothAP=0.2 · lr=2e-4 · LHP=true (như v3c).
- **VAL-B** split-by-video seed-42 (≈621 query) → chọn **best.pth**.

🔴 **GPU & batch (đã tính):** trainer **CHỈ DÙNG 1 GPU** (không DataParallel/DDP) → dù chọn **T4×2 thì GPU thứ 2 nằm không**, batch vẫn giới hạn theo **1×T4 (15GB)**:
- **batch 20** + frozen + pose ≈ **13.5G/15G** → ✅ an toàn (đã đo ở v3c).
- **batch 32** ≈ **~21G > 15G → OOM** trên 1 GPU (grad-checkpointing chưa wire). → **dùng 20**, muốn "batch lớn" thì tăng `grad_accum` (không tốn VRAM).

⏱️ **Thời gian (đã tính): 10 epoch trên 50K ≈ 16-19h → KHÔNG thể 1 commit** (9h; quá giờ là **MẤT TRẮNG output**). Notebook **tự tính & cắp epoch** cho vừa ~8h. *(50K gấp 5× data → ~4 epoch đã thấy nhiều data hơn cả 10 epoch ở 10K.)*

**Add Input:** dataset chứa `train_50k_hard_data.tar.zst` + `xvlm_16m_base.th`. GPU **T4** (đơn là đủ) · Internet ON.

In [ ]:
# [1/7] GPU + clone/pull repo (can ban MOI: PairBatchSampler + FIX bool override)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Bat GPU T4 trong Settings!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
assert "PairBatchSampler" in pathlib.Path("src/star/data/sampler.py").read_text()
assert 'low == "true"' in pathlib.Path("src/star/config.py").read_text(), "thieu FIX bool override -> pull lai!"
print("repo OK:", os.getcwd())

In [ ]:
# [2/7] Env pinned + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/7] Tim + giai nen 50K (tar.zst) + base ckpt
import glob, os, pathlib, shutil
def find_one(p):
    h = sorted(set(glob.glob(f"/kaggle/input/*/{p}") + glob.glob(f"/kaggle/input/*/**/{p}", recursive=True)))
    return h[0] if h else None

CKPT = find_one("xvlm_16m_base.th")
ARCH = find_one("train_50k_hard_data.tar.zst")
JSONL = find_one("train_50k_hard.jsonl")
marker = "/kaggle/working/extract"
got = glob.glob(f"{marker}/**/train_50k_hard.jsonl", recursive=True)
if JSONL:
    pass
elif got:
    pass
else:
    assert ARCH, "Khong thay train_50k_hard_data.tar.zst — Add Input dataset 50K!"
    os.makedirs(marker, exist_ok=True)
    print("giai nen 50K (~5-8 phut)...")
    if shutil.which("zstd"):
        os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
    else:
        import zstandard, tarfile
        with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
            with tarfile.open(fileobj=zr, mode="r|") as tf:
                tf.extractall(marker)
got = glob.glob("/kaggle/input/**/train_50k_hard.jsonl", recursive=True) + \
      glob.glob(f"{marker}/**/train_50k_hard.jsonl", recursive=True)
assert got, "giai nen that bai / khong thay train_50k_hard.jsonl"
DATA_ROOT = str(pathlib.Path(got[0]).parents[2])
if not CKPT:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(CKPT).stat().st_size > 8e8
import shutil as _sh
print("DATA_ROOT =", DATA_ROOT, "| CKPT =", CKPT)
print("disk /kaggle/working free:", round(_sh.disk_usage('/kaggle/working').free/2**30, 1), "GB")

In [ ]:
# [4/7] Build manifest (GIONG HET v3c: keypoints + bbox + pair_image_id + split video seed-42)
import json, glob, pathlib
import numpy as np, pandas as pd
root = pathlib.Path(DATA_ROOT)
JSONL = glob.glob(f"{DATA_ROOT}/**/train_*hard.jsonl", recursive=True)[0]
POSEJ = glob.glob(f"{DATA_ROOT}/**/*vitpose*.json", recursive=True)[0]
print("jsonl:", JSONL, "\nvitpose:", POSEJ)

anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")
for line in open(JSONL, encoding="utf-8"):
    r = json.loads(line); anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"],
        pair_image_id=r.get("hard_i_id")))
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid, pair_image_id=None)
df = pd.DataFrame(anchors + [v for k, v in hard_rows.items() if k not in anchor_ids])
pose = json.load(open(POSEJ, encoding="utf-8"))["items"]
def kpts_of(iid):
    it = pose.get(iid)
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None
def bbox_of(iid):
    it = pose.get(iid); b = it.get("primary_bbox_norm_xyxy") if it else None
    if not b:
        return None
    x1, y1, x2, y2 = b
    return [x1, y1, max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)]
df["keypoints"] = df.image_id.map(kpts_of); df["bbox"] = df.image_id.map(bbox_of)
rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""
MANIFEST = "/kaggle/working/manifest_50k_hard.parquet"; df.to_parquet(MANIFEST, index=False)
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
print(f"rows={len(df)} train={(df.split=='train').sum()} "
      f"valb_q={((df.split=='valb') & (df.caption!='')).sum()} kpts={df.keypoints.notna().mean():.0%} leakage={leak}")
assert leak == 0 and df.keypoints.notna().mean() > 0.99

In [ ]:
# [5/7] CAU HINH v3c + TU TINH EPOCHS cho vua ~8h (tranh qua 9h mat output)
BATCH = 20    # single-GPU -> 20 an toan (~13.5G). 32 = OOM (xem markdown). Muon batch lon -> tang grad_accum.
import pandas as pd
dfm = pd.read_parquet(MANIFEST)
PAIRS = int(dfm[dfm.split == "train"].pair_image_id.notna().sum())
bpe = PAIRS / max(BATCH // 2, 1)                 # PairBatchSampler: BATCH//2 cap/batch
epm = bpe * 1.6 / 60                              # ~1.6 s/batch (do tu v3c, frozen+pose)
EPOCHS = max(2, min(10, int(480 / max(epm, 1)))) # ngan sach ~8h train -> tong <9h
print(f"pairs(train)={PAIRS} | ~{bpe:.0f} batch/epoch | ~{epm:.1f} min/epoch")
print(f"=> EPOCHS = {EPOCHS}  (~{EPOCHS*epm/60:.1f}h train; 10 epoch se ~{10*epm/60:.1f}h -> qua 9h nen cat)")

DATA = f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT}"
V3C = (f"data.group_by=pair model.lora_freeze_text=true model.pose_enabled=true "
       f"data.lhp_enabled=true loss.lambda_smooth_ap=0.2 optim.lr_lora=2e-4 "
       f"optim.epochs={EPOCHS} train.early_stop_patience=6 train.eval_every_epochs=0.5 "
       f"train.batch_size={BATCH} train.grad_accum=2 train.grad_norm_every=0 train.out_dir=/kaggle/working/v3c50k")
print("\nCONFIG:", V3C)

In [ ]:
# [6/7] GATE overfit-one-batch (probe 'vram='). OOM -> ha BATCH=16 o cell 5
gate = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch --set {DATA} {V3C}"
print(gate)
!{gate}
print("Neu OOM: dat BATCH=16 (cell 5) chay lai. (BATCH=32 chac chan OOM tren 1 GPU.)")

In [ ]:
# [7/7] TRAIN v3c-50K -> best.pth (VAL-B chon checkpoint). Phai HOAN TAT <9h moi co output!
import pathlib, time, torch
cmd = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --set {DATA} {V3C}"
print(cmd); t0 = time.time()
!{cmd}
mins = round((time.time() - t0) / 60, 1)
BEST = "/kaggle/working/v3c50k/best.pth"
assert pathlib.Path(BEST).exists(), "train fail — xem log (OOM? ha BATCH=16)"
raw = torch.load(BEST, map_location="cpu", weights_only=False)
print("=" * 70)
print(f"XONG ({mins} phut): VAL-B best mAP = {raw.get('best_metric'):.4f}")
print(f"  report: {(raw.get('extra') or {}).get('report')}")
print(f"  moc v3c(10K)=0.7485 | best.pth = {BEST} ({pathlib.Path(BEST).stat().st_size/1e6:.0f} MB)")
del raw
!ls -lh /kaggle/working/v3c50k/best.pth

## Lấy kết quả & ghi chú
- **best.pth** ở `/kaggle/working/v3c50k/best.pth` → **Save Version → Output** để tải. (Chỉ cần file này, không inference.)
- **Phải Save & Run All (Commit)** để chạy nền/đi ngủ; notebook đã tự cắp epoch cho **<9h** → commit hoàn tất, output được lưu. *(Nếu để epoch quá tay mà commit bị cắt ở 9h → MẤT output.)*
- **Muốn đủ 10 epoch thật** (50K ~16-19h) → cần **resume nhiều commit** (lưu optimizer+step rồi load tiếp) — hiện chưa có, báo mình thêm.
- **batch**: giữ 20. 32 OOM trên 1 GPU. Muốn batch hiệu dụng lớn hơn → tăng `train.grad_accum` (vd 3 → eff. 60), KHÔNG tốn VRAM, nhưng KHÔNG nhanh hơn.
- **Disk**: 50K giải nén ~2-4GB + checkpoint ~0.86GB → vừa /kaggle/working. Không cần xoá gì.